# Baselines — Progress Review 2 (Đại)

**Task:** Twitter account bot detection (bot = 1, human = 0), MGTAB 10,199 accounts. **Evaluation:** shared `split()` in `mgtab.py`; validation only. Never use stance as a model input or test labels to tune models.

**Feature groups:** P (16 profile), C (4 profile counts), T (768 tweet embeddings), G (28 graph-computed). Compare **P+C+T** with **P+C+T+G** to isolate the contribution of computed graph features.

**Node2Vec:** PyG Node2Vec with a custom Numba second-order biased-walk sampler (`embeddings.py`), 128-dimensional embeddings, 10 walks per node, walk length 80, window size 10, 5 epochs per (p,q), p/q in {0.5, 1, 2}. This notebook includes the **recorded 9-run grid from the completed Colab run**, so routine validation does **not** repeat ~hours of training. To retrain Node2Vec, set the flag in its section.

**Reproducibility note:** CPU/GPU, installation versions, random-walk sampling, and MLP training can lead to small variation on rerun. Node2Vec checkpoints restore weights and optimizer but not all random state. The retained code-cell outputs below are from the actual recorded run, not new results produced by this clean copy. The supplied CSVs are the archived validation results.

In [1]:
# 1. Setup (Colab or local)
import os, sys
if not os.path.exists("mgtab.py"):
    if "google.colab" in sys.modules:
        if not os.path.exists("sna-content-moderation"):
            !git clone -q https://github.com/ThSonDev/sna-content-moderation.git
        %cd sna-content-moderation
        !pip install -q -r requirements.txt
        !python install_pyg.py
    else:
        %cd ..
sys.path.insert(0, os.getcwd())

/content/sna-content-moderation
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
PyG ready 2.8.1


In [2]:
# 2. Download MGTAB (31 MB) into data/MGTAB, only once
import urllib.request, zipfile
if not os.path.exists("data/MGTAB/features.pt"):
    os.makedirs("data", exist_ok=True)
    url = "https://drive.usercontent.google.com/download?id=1gbWNOoU1JB8RrTu2a5j9KMNVa9wX72Fe&export=download&confirm=t"
    urllib.request.urlretrieve(url, "data/MGTAB.zip")
    zipfile.ZipFile("data/MGTAB.zip").extractall("data")

In [3]:
# 3. Load data and the shared split (never make your own split)
import numpy as np
from mgtab import load, relations, split, best_threshold, evaluate, RELATIONS
X, y, stance, E, T, W = load()
rel = relations(E, T, len(y))   # {"follow": (src, dst), ...}
train, val, test = split(y)     # test stays locked until the final run
print(X.shape, {r: len(s) for r, (s, d) in rel.items()})

(10199, 788) {'follow': 425863, 'mention': 110868, 'reply': 22327, 'quote': 20901, 'url': 263800, 'hashtag': 300000}


## 1. Feature groups & Fair Logistic Regression

Fit scalers on the **training accounts only**, and keep raw T embeddings as in the team's original configuration. Select LR regularization on validation PR-AUC.

In [4]:

# ============================================================
# Progress Review 2 - Fair Baseline (P + C + T)
# ============================================================

import numpy as np
import pandas as pd
from features import feature_groups
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

# 1. Split features into four groups
groups, g_names = feature_groups(X, rel)

for name, data in groups.items():
    print(f"{name}: {data.shape}")

# 2. Standardize P and C using train statistics only
scaled = {}
for name in ["P", "C", "G"]:
    scaler = StandardScaler()
    scaler.fit(groups[name][train])
    scaled[name] = scaler.transform(groups[name])

# T: retain original tweet embeddings
scaled["T"] = groups["T"]

# 3. Fair baseline features: P + C + T
Z_fair = np.hstack([
    scaled["P"],
    scaled["C"],
    scaled["T"]
])

assert Z_fair.shape == (10199, 788)

# 4. Train Logistic Regression and select C on validation
best = None

for C in [0.01, 0.1, 1, 10]:
    model = LogisticRegression(
        C=C,
        class_weight="balanced",
        max_iter=5000
    )
    model.fit(Z_fair[train], y[train])

    scores = model.predict_proba(Z_fair[val])[:, 1]
    pr_auc = average_precision_score(y[val], scores)

    print(f"C={C}, PR-AUC={pr_auc:.4f}")

    if best is None or pr_auc > best[0]:
        best = (pr_auc, C, scores)

# 5. Evaluate using validation only
best_pr_auc, best_C, best_scores = best
threshold = best_threshold(y[val], best_scores)
metrics = evaluate(y[val], best_scores, threshold)

print("\n===== FAIR BASELINE RESULTS =====")
print("Best C:", best_C)
print("PR-AUC:", round(metrics["pr_auc"], 4))
print("Macro-F1:", round(metrics["macro_f1"], 4))
print("Accuracy:", round(metrics["accuracy"], 4))
print("P@100:", round(metrics["p@100"], 4))
print("P@200:", round(metrics["p@200"], 4))
print("Threshold:", round(metrics["threshold"], 4))


P: (10199, 16)
C: (10199, 4)
T: (10199, 768)
G: (10199, 28)
C=0.01, PR-AUC=0.8364
C=0.1, PR-AUC=0.8767
C=1, PR-AUC=0.8759
C=10, PR-AUC=0.8781

===== FAIR BASELINE RESULTS =====
Best C: 10
PR-AUC: 0.8781
Macro-F1: 0.8858
Accuracy: 0.9072
P@100: 0.96
P@200: 0.915
Threshold: 0.5872


## 2. Graph-only controls (G, G+C) & Fusion LR

Same LR grid and validation threshold selection for each model.

In [5]:

# ============================================================
# Progress Review 2 - Graph-only comparisons
# G-only (28) vs G + C (32)
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
import numpy as np

def run_graph_lr(Z, model_name):
    best = None

    for C in [0.01, 0.1, 1, 10]:
        model = LogisticRegression(
            C=C,
            class_weight="balanced",
            max_iter=5000
        )
        model.fit(Z[train], y[train])

        scores = model.predict_proba(Z[val])[:, 1]
        ap = average_precision_score(y[val], scores)

        if best is None or ap > best[0]:
            best = (ap, C, scores)

    best_ap, best_C, best_scores = best
    threshold = best_threshold(y[val], best_scores)
    metrics = evaluate(y[val], best_scores, threshold)

    print(f"\n===== {model_name} =====")
    print("Best C:", best_C)
    print("PR-AUC:", round(metrics["pr_auc"], 4))
    print("Macro-F1:", round(metrics["macro_f1"], 4))
    print("Accuracy:", round(metrics["accuracy"], 4))
    print("P@100:", round(metrics["p@100"], 4))
    print("P@200:", round(metrics["p@200"], 4))
    print("Threshold:", round(metrics["threshold"], 4))

    return {
        "model": model_name,
        "C": best_C,
        **metrics
    }


# G-only: 28 computed graph features
Z_g = scaled["G"]

result_g_only = run_graph_lr(
    Z_g,
    "B2a G-only (28)"
)

# G + C: 28 graph features + 4 profile counts
Z_gc = np.hstack([
    scaled["G"],
    scaled["C"]
])

assert Z_gc.shape == (10199, 32)

result_g_counts = run_graph_lr(
    Z_gc,
    "B2a G + counts (32)"
)

# Preliminary comparison
print("\n===== GRAPH COMPARISON =====")
print(
    "G-only PR-AUC:",
    round(result_g_only["pr_auc"], 4)
)
print(
    "G + counts PR-AUC:",
    round(result_g_counts["pr_auc"], 4)
)



===== B2a G-only (28) =====
Best C: 0.1
PR-AUC: 0.707
Macro-F1: 0.7701
Accuracy: 0.8052
P@100: 0.81
P@200: 0.765
Threshold: 0.6008

===== B2a G + counts (32) =====
Best C: 0.1
PR-AUC: 0.7594
Macro-F1: 0.834
Accuracy: 0.8608
P@100: 0.8
P@200: 0.82
Threshold: 0.5971

===== GRAPH COMPARISON =====
G-only PR-AUC: 0.707
G + counts PR-AUC: 0.7594


In [6]:

# ============================================================
# Progress Review 2 - P1 Fusion-LR (P + C + T + G)
# ============================================================

# Combine all four feature groups
Z_fusion = np.hstack([
    scaled["P"],
    scaled["C"],
    scaled["T"],
    scaled["G"]
])

assert Z_fusion.shape == (10199, 816)

# Use the same LR training and evaluation settings
result_fusion = run_graph_lr(
    Z_fusion,
    "P1 Fusion-LR (P+C+T+G)"
)

# Compare against the fair baseline
fair_pr_auc = metrics["pr_auc"]
fusion_pr_auc = result_fusion["pr_auc"]

gain = fusion_pr_auc - fair_pr_auc

print("\n===== FAIR COMPARISON =====")
print(f"Fair baseline P+C+T: {fair_pr_auc:.4f}")
print(f"P1 Fusion P+C+T+G: {fusion_pr_auc:.4f}")
print(f"Graph gain (PR-AUC): {gain:+.4f}")

# Store intermediate results for the final CSV
result_fair = {
    "model": "B1 Fair LR (P+C+T)",
    "C": best_C,
    **metrics
}

print("\nComparison complete.")



===== P1 Fusion-LR (P+C+T+G) =====
Best C: 0.1
PR-AUC: 0.9152
Macro-F1: 0.9064
Accuracy: 0.9235
P@100: 0.97
P@200: 0.945
Threshold: 0.646

===== FAIR COMPARISON =====
Fair baseline P+C+T: 0.8781
P1 Fusion P+C+T+G: 0.9152
Graph gain (PR-AUC): +0.0371

Comparison complete.


## 3. MLP — P+C+T

Two hidden layers, weighted cross-entropy, 8 validation-selected combinations; early stop on validation PR-AUC.

In [7]:

# ============================================================
# B1 MLP - Fair baseline P + C + T
# ============================================================

import copy
import random
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)

# Reuse the fair feature matrix
X_train = torch.tensor(
    Z_fair[train], dtype=torch.float32, device=device
)
X_val = torch.tensor(
    Z_fair[val], dtype=torch.float32, device=device
)

y_train = torch.tensor(
    y[train], dtype=torch.long, device=device
)

# Class weights calculated on train only
counts = np.bincount(y[train], minlength=2)
weights = len(train) / (2.0 * counts)
class_weights = torch.tensor(
    weights, dtype=torch.float32, device=device
)

class MLP(nn.Module):
    def __init__(self, input_dim, hidden, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 2)
        )

    def forward(self, x):
        return self.net(x)


def train_mlp(hidden, dropout, lr, max_epochs=200, patience=20):
    torch.manual_seed(SEED)

    model = MLP(
        Z_fair.shape[1], hidden, dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr
    )
    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    best_ap = -1.0
    best_state = None
    best_scores = None
    best_epoch = 0
    wait = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()

        logits = model(X_train)
        loss = criterion(logits, y_train)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val)
            scores = torch.softmax(
                val_logits, dim=1
            )[:, 1].cpu().numpy()

        ap = average_precision_score(
            y[val], scores
        )

        if ap > best_ap + 1e-6:
            best_ap = ap
            best_epoch = epoch
            best_state = copy.deepcopy(
                model.state_dict()
            )
            best_scores = scores.copy()
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            break

    model.load_state_dict(best_state)

    return {
        "hidden": hidden,
        "dropout": dropout,
        "lr": lr,
        "best_epoch": best_epoch,
        "pr_auc": best_ap,
        "scores": best_scores
    }


# Grid search: 2 x 2 x 2 = 8 configurations
mlp_runs = []

for hidden in [64, 128]:
    for dropout in [0.3, 0.5]:
        for lr in [0.005, 0.01]:
            run = train_mlp(
                hidden=hidden,
                dropout=dropout,
                lr=lr
            )
            mlp_runs.append(run)

            print(
                f"hidden={hidden}, "
                f"dropout={dropout}, "
                f"lr={lr}, "
                f"epoch={run['best_epoch']}, "
                f"PR-AUC={run['pr_auc']:.4f}"
            )

# Select best validation PR-AUC
best_mlp = max(
    mlp_runs, key=lambda r: r["pr_auc"]
)

mlp_threshold = best_threshold(
    y[val], best_mlp["scores"]
)

mlp_metrics = evaluate(
    y[val],
    best_mlp["scores"],
    mlp_threshold
)

result_mlp = {
    "model": "B1 MLP (P+C+T)",
    "C": np.nan,
    "hidden": best_mlp["hidden"],
    "dropout": best_mlp["dropout"],
    "lr": best_mlp["lr"],
    "best_epoch": best_mlp["best_epoch"],
    **mlp_metrics
}

print("\n===== BEST MLP RESULTS =====")
for k, v in result_mlp.items():
    print(f"{k}: {v}")


Device: cuda
hidden=64, dropout=0.3, lr=0.005, epoch=187, PR-AUC=0.8953
hidden=64, dropout=0.3, lr=0.01, epoch=197, PR-AUC=0.8881
hidden=64, dropout=0.5, lr=0.005, epoch=160, PR-AUC=0.8907
hidden=64, dropout=0.5, lr=0.01, epoch=137, PR-AUC=0.8888
hidden=128, dropout=0.3, lr=0.005, epoch=118, PR-AUC=0.8938
hidden=128, dropout=0.3, lr=0.01, epoch=122, PR-AUC=0.8931
hidden=128, dropout=0.5, lr=0.005, epoch=178, PR-AUC=0.9021
hidden=128, dropout=0.5, lr=0.01, epoch=113, PR-AUC=0.8780

===== BEST MLP RESULTS =====
model: B1 MLP (P+C+T)
C: nan
hidden: 128
dropout: 0.5
lr: 0.005
best_epoch: 178
pr_auc: 0.9020985877758537
roc_auc: 0.9695093092728
macro_f1: 0.8993038772269024
accuracy: 0.9202614379084967
p@100: 0.94
p@200: 0.955
threshold: 0.7623680830001831


## 4. Node2Vec — recorded grid & optional reproduction

Graph topology is merged, undirected, deduplicated and **transductive/unlabeled**. Node2Vec learns from edges; LR uses bot labels of `train` only. The `results/node2vec_grid.csv` file records the **nine previously completed 5-epoch runs**. `results/baselines.csv` records the complete validation metrics of their best configuration.

**Normal execution:** loads recorded results and skips expensive retraining. **To recompute:** set `RUN_NODE2VEC_GRID = True` in the next cell (hours of compute; checkpoints under `results/node2vec_runs/`; keep those binary artifacts out of Git). Choose p/q by validation PR-AUC. No personal Google Drive paths are required.

In [ ]:
# Node2Vec reproducible orchestration (normal run uses archived results)
from pathlib import Path
from embeddings import merged_edge_index, train_node2vec
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
import pandas as pd
import numpy as np

RUN_NODE2VEC_GRID = False  # Set True to retrain nine configurations (expensive)
GRID_CSV = Path("results/node2vec_grid.csv")
RUN_DIR = Path("results/node2vec_runs")
REFERENCE_RESULTS = Path("results/baselines.csv")

edge_index = merged_edge_index(rel, len(y))
print("Merged undirected graph:", tuple(edge_index.shape))

# Read the existing summary BEFORE the final CSV is overwritten.
# It contains metrics beyond PR-AUC/Macro-F1 for the completed Node2Vec grid.
reference = pd.read_csv(REFERENCE_RESULTS)
n2v_reference = reference.loc[
    reference["model"] == "B2b Node2Vec (best p/q)"
]
if len(n2v_reference) != 1:
    raise ValueError("Expected one recorded best Node2Vec row in results/baselines.csv")

def evaluate_n2v(Z, p, q):
    scaler = StandardScaler().fit(Z[train])
    Z_scaled = scaler.transform(Z)
    best = None
    for C_value in [0.01, 0.1, 1, 10]:
        lr_model = LogisticRegression(C=C_value, class_weight="balanced", max_iter=5000)
        lr_model.fit(Z_scaled[train], y[train])
        s = lr_model.predict_proba(Z_scaled[val])[:, 1]
        ap = average_precision_score(y[val], s)
        if best is None or ap > best[0]:
            best = (ap, C_value, s.copy())
    _, C_value, s = best
    threshold = best_threshold(y[val], s)
    m = evaluate(y[val], s, threshold)
    return {"model":"B2b Node2Vec (best p/q)", "p":p, "q":q, "epochs":5, "C":C_value, **m}

if RUN_NODE2VEC_GRID:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    grid_records = []
    best_n2v = None
    for p in [0.5, 1.0, 2.0]:
        for q in [0.5, 1.0, 2.0]:
            print(f"Training p={p}, q={q}")
            Z, history = train_node2vec(
                edge_index, len(y), p, q, output_dir=str(RUN_DIR),
                epochs=5, batch_size=256, lr=0.01, seed=42,
            )
            result = evaluate_n2v(Z, p, q)
            grid_records.append(result)
            if best_n2v is None or result["pr_auc"] > best_n2v["pr_auc"]:
                best_n2v = result
            pd.DataFrame(grid_records).to_csv(GRID_CSV, index=False)
else:
    if not GRID_CSV.exists():
        raise FileNotFoundError(f"Missing archived Node2Vec grid: {GRID_CSV}")
    grid_records = pd.read_csv(GRID_CSV).to_dict("records")
    best_n2v = n2v_reference.iloc[0].dropna().to_dict()

n2v_grid = pd.DataFrame(grid_records).sort_values(["p", "q"]).reset_index(drop=True)
assert len(n2v_grid) == 9, "Expected nine p/q configurations"
selected = n2v_grid.loc[n2v_grid["pr_auc"].idxmax()]
assert np.isclose(float(best_n2v["p"]), float(selected["p"]))
assert np.isclose(float(best_n2v["q"]), float(selected["q"]))
print("Best Node2Vec p/q:", (float(selected["p"]), float(selected["q"])))
print("Best validation PR-AUC:", float(best_n2v["pr_auc"]))
display(n2v_grid[["p", "q", "epochs", "C", "pr_auc", "macro_f1"]])


## 5. Majority & Label Propagation

Majority always predicts human. Label propagation clamps **training labels only**, uses the unlabeled merged graph, and evaluates on validation.

In [34]:

# ============================================================
# Progress Review 2 - B0 Majority and B3 Label Propagation
# ============================================================

import numpy as np
from scipy.sparse import coo_matrix

# ------------------------------------------------------------
# B0: Always predict human (score = 0)
# ------------------------------------------------------------

scores_b0 = np.zeros(len(val), dtype=float)

metrics_b0 = evaluate(
    y[val],
    scores_b0,
    0.5
)

result_b0 = {
    "model": "B0 Majority",
    "C": np.nan,
    **metrics_b0
}

print("===== B0 MAJORITY =====")
print("PR-AUC:", round(result_b0["pr_auc"], 4))
print("Macro-F1:", round(result_b0["macro_f1"], 4))


# ------------------------------------------------------------
# B3: Label Propagation on merged undirected graph
# ------------------------------------------------------------

n = len(y)

# edge_index was created by merged_edge_index(rel, len(y))
src = edge_index[0].cpu().numpy()
dst = edge_index[1].cpu().numpy()

A = coo_matrix(
    (
        np.ones(len(src), dtype=np.float64),
        (src, dst)
    ),
    shape=(n, n)
).tocsr()

# Each graph connection contributes one vote
A.data[:] = 1.0
A.setdiag(0)
A.eliminate_zeros()

degree = np.asarray(A.sum(axis=1)).ravel()
isolated = degree == 0

# Train labels are the only observed labels
prior = float(np.mean(y[train]))

scores_lp = np.full(n, prior, dtype=np.float64)
scores_lp[train] = y[train]

max_iter = 100
tol = 1e-6

for iteration in range(max_iter):
    updated = A @ scores_lp

    connected = ~isolated
    updated[connected] /= degree[connected]
    updated[isolated] = prior

    # Clamp the train labels after each iteration
    updated[train] = y[train]

    diff = np.max(np.abs(updated - scores_lp))
    scores_lp = updated

    if diff < tol:
        break

# Evaluate validation only
val_scores_lp = scores_lp[val]

threshold_lp = best_threshold(
    y[val],
    val_scores_lp
)

metrics_lp = evaluate(
    y[val],
    val_scores_lp,
    threshold_lp
)

result_b3 = {
    "model": "B3 Label Propagation",
    "C": np.nan,
    **metrics_lp
}

print("\n===== B3 LABEL PROPAGATION =====")
print("Iterations:", iteration + 1)
print("Isolated nodes:", int(isolated.sum()))
print("Train bot prior:", round(prior, 4))
print("PR-AUC:", round(result_b3["pr_auc"], 4))
print("Macro-F1:", round(result_b3["macro_f1"], 4))
print("Threshold:", round(threshold_lp, 4))


===== B0 MAJORITY =====
PR-AUC: 0.2693
Macro-F1: 0.4222

===== B3 LABEL PROPAGATION =====
Iterations: 12
Isolated nodes: 54
Train bot prior: 0.2695
PR-AUC: 0.5283
Macro-F1: 0.6845
Threshold: 0.2273


## 6. Final validation results

Recompute LR/MLP/B0/B3 from the active session and include the recorded or optionally retrained best Node2Vec. **No test metrics are reported.**

In [35]:
# Export all eight baseline/control rows with a fixed column schema.
import os
import pandas as pd

all_results = [
    result_b0,
    result_fair,           # Selected C is the one found on validation, not hardcoded.
    result_g_only,
    result_g_counts,
    result_fusion,
    result_mlp,
    {**best_n2v, "model": "B2b Node2Vec (best p/q)"},
    result_b3,
]
columns = [
    "model", "C", "p", "q", "epochs", "hidden", "dropout", "lr", "best_epoch",
    "pr_auc", "roc_auc", "macro_f1", "accuracy", "p@100", "p@200", "threshold",
]
df_final = pd.DataFrame(all_results).reindex(columns=columns)
os.makedirs("results", exist_ok=True)
output_path = "results/baselines.csv"
df_final.to_csv(output_path, index=False)
print("===== FINAL VALIDATION RESULTS =====")
display(df_final[["model", "pr_auc", "macro_f1", "accuracy"]])
print("Saved:", output_path, "| Rows:", len(df_final))


===== FINAL BASELINE RESULTS =====


,model,pr_auc,macro_f1,accuracy
0,B0 Majority,0.269281,0.422205,0.730719
1,B1 Fair LR (P+C+T),0.878092,0.885812,0.907190
2,B2a G-only (28),0.707009,0.770110,0.805229
3,B2a G + counts (32),0.759382,0.833956,0.860784
4,P1 Fusion-LR (P+C+T+G),0.915179,0.906369,0.923529
5,B1 MLP (P+C+T),0.902099,0.899304,0.920261
6,B2b Node2Vec (best p/q),0.456357,0.607585,0.690196
7,B3 Label Propagation,0.528349,0.684480,0.744444



Rows: 8
Saved: results/baselines.csv
Backup: /content/drive/MyDrive/SNA-ProgressReview2/baselines.csv
